# Iteración 2 — Reglas de asociación (Apriori + FP-Growth)

Objetivo del proyecto: **descubrir qué características (features) se asocian con el éxito** de un videojuego, donde el éxito se mide por:
- reseñas (calidad y volumen)
- ventas
- Peak CCU

En este notebook usaremos reglas del tipo:
- **LHS (features)** → **RHS (éxito)**

Notas de rendimiento:
- El CSV completo tiene ~114k filas y muchas variables; por defecto se carga un **sample** para iterar rápido.
- **Apriori** puede consumir mucha RAM; por eso lo corremos con un subconjunto de items y `low_memory=True`.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)
pd.set_option('display.max_colwidth', 80)

In [3]:
# Carga: pon NROWS=None para todo el dataset (más lento / más RAM)
# Por defecto usamos un sample más chico para que Apriori/FP-Growth terminen rápido.
NROWS = None

df = pd.read_csv("data_categorizado.csv", nrows=NROWS, low_memory=False)
df.shape

(114174, 131)

In [4]:
# Vista general rápida
display(df.head(3))
print('Columnas:', len(df.columns))
df.columns[:25]

,appID,Name,Release date,Estimated owners,Peak CCU,Price,Discount,DLC count,About the game,Supported languages,Full audio languages,Reviews,Windows,Mac,Linux,Metacritic score,Positive,Negative,Achievements,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Price_category,Release_datetime,...,tag_Indie,tag_Linear,tag_Minimalist,tag_Multiplayer,tag_Multiple Endings,tag_Mystery,tag_Open World,tag_Physics,tag_Pixel Graphics,tag_Platformer,tag_Point & Click,tag_Psychological Horror,tag_Puzzle,tag_PvE,tag_RPG,tag_Realistic,tag_Relaxing,tag_Retro,tag_Sci-fi,tag_Shooter,tag_Simulation,tag_Singleplayer,tag_Story Rich,tag_Strategy,tag_Stylized,tag_Survival,tag_Third Person,tag_Top-Down,tag_VR,tag_Visual Novel
0,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,5.24,65,0,"Springtime, April: when the cherry trees come into full bloom. The protagoni...",['English'],[],NaN,True,False,False,0,252,3,0,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,Family Sharing",Adventure,"Adventure,Visual Novel,Anime,Cute",5.00-9.99,2016-07-29,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,4.99,0,0,"Immerse yourself in the most beloved, mystical and entrancing world of Edgar...","['English', 'French', 'German', 'Russian']",[],NaN,True,True,False,0,21,3,0,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Object,2D,Colorful,Stylized,Logic,M...",0.01-4.99,2019-05-06,...,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0
2,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' It's been three years since I'...",['Korean'],['Korean'],NaN,True,False,False,0,0,0,19,0,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,5.00-9.99,2024-10-31,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Columnas: 131


Index(['appID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Price', 'Discount', 'DLC count', 'About the game',
       'Supported languages', 'Full audio languages', 'Reviews', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Positive', 'Negative',
       'Achievements', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks',
       'Developers', 'Publishers'],
      dtype='str')

In [4]:
# Variables one-hot disponibles para minería
genre_cols = [c for c in df.columns if c.startswith('genre_')]
tag_cols = [c for c in df.columns if c.startswith('tag_')]
print('N genre_*:', len(genre_cols))
print('N tag_*  :', len(tag_cols))

ohe_cols = genre_cols + tag_cols
if ohe_cols:
    support = df[ohe_cols].fillna(0).astype(int).clip(0, 1).mean().sort_values(ascending=False)
    display(support.head(20).to_frame('support_rate'))
else:
    print('No se detectaron columnas genre_*/tag_* en este CSV.')

N genre_*: 33
N tag_*  : 59


,support_rate
genre_Indie,0.704500
tag_Singleplayer,0.447833
genre_Casual,0.443233
tag_Indie,0.430200
genre_Action,0.402300
genre_Adventure,0.391667
tag_Casual,0.327633
tag_Action,0.319200
tag_Adventure,0.305433
tag_2D,0.240267


## Selección de columnas para reglas relevantes (según tu objetivo)

### Consecuentes (RHS = éxito)
Usaremos items `target=...` construidos desde:
- `review_score_cat` → `target=review_good` (calidad)
- `total_reviews` → `target=reviews_50p_plus` / `target=reviews_75p_plus` (volumen)
- `sales_cat` → `target=sales_good`
- `ccu_cat` → `target=ccu_good`

### Antecedentes (LHS = características)
Items construidos desde columnas que son interpretables y útiles para *predecir* o perfilar éxito:
- `genre_*` y `tag_*` (contenido / estilo)
- `Categories` (modo de juego / features de Steam: Single-player, Multiplayer, VR, etc.)
- `Price_category` (estrategia de precio)
- `Release_season` y `Release_year` agrupado en `Release_era` (contexto de mercado)
- `Windows`/`Mac`/`Linux` (alcance de plataforma)
- `DLC count` (en bins) y `Discount` (en bins; opcional)

En reglas, lo más útil para tu informe suele ser: **LHS pequeño (2–3 items) → RHS de éxito**, ordenado por `lift` y `confidence`.

In [5]:
# Algoritmos de asociación (Apriori + FP-Growth)
try:
    from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        'Falta instalar mlxtend. En una terminal: pip install mlxtend',
    ) from e

In [6]:
def _bin_series(s: pd.Series, bins, labels=None) -> pd.Series:
    s_num = pd.to_numeric(s, errors='coerce')
    return pd.cut(s_num, bins=bins, labels=labels, include_lowest=True)

def build_item_matrix(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str], list[str]]:
    # -------- Targets (RHS) --------
    targets = {}

    if 'review_score_cat' in df.columns:
        good_reviews = {'Mostly Positive', 'Very Positive', 'Overwhelmingly Positive'}
        targets['target=review_good'] = df['review_score_cat'].isin(good_reviews)

    if 'sales_cat' in df.columns:
        good_sales = {'Popular', 'Very Popular', 'Top Game', 'Blockbuster', 'Mega Hit'}
        targets['target=sales_good'] = df['sales_cat'].isin(good_sales)

    if 'ccu_cat' in df.columns:
        good_ccu = {'High', 'Very High'}
        targets['target=ccu_good'] = df['ccu_cat'].isin(good_ccu)

    if 'total_reviews' in df.columns:
        tr = pd.to_numeric(df['total_reviews'], errors='coerce').fillna(0)
        q50 = float(tr.quantile(0.5))
        q75 = float(tr.quantile(0.75))
        targets['target=reviews_50p_plus'] = tr >= q50
        targets['target=reviews_75p_plus'] = tr >= q75

    targets_df = pd.DataFrame(targets).fillna(False).astype(bool) if targets else pd.DataFrame(index=df.index)
    target_items = list(targets_df.columns)

    # -------- Features (LHS) --------
    feature_frames = []

    # genre_*/tag_* (one-hot ya preparado)
    genre_cols = [c for c in df.columns if c.startswith('genre_')]
    tag_cols = [c for c in df.columns if c.startswith('tag_')]

    if genre_cols:
        g = df[genre_cols].fillna(0).astype(int).clip(0, 1)
        g = g.rename(columns=lambda c: c.replace('genre_', 'genre='))
        feature_frames.append(g.astype(bool))

    if tag_cols:
        t = df[tag_cols].fillna(0).astype(int).clip(0, 1)
        t = t.rename(columns=lambda c: c.replace('tag_', 'tag='))
        feature_frames.append(t.astype(bool))

    # Categories (lista separada por coma)
    if 'Categories' in df.columns:
        raw = df['Categories'].fillna('').astype(str).str.split(',')
        cleaned = raw.apply(lambda xs: [x.strip() for x in xs if x and x.strip()])
        flat = [x for xs in cleaned.tolist() for x in xs]
        if flat:
            counts = pd.Series(flat).value_counts()
            support = counts / len(df)
            selected = support[(support >= 0.01) & (support <= 0.60)].sort_values(ascending=False).head(30).index
            cat_df = pd.DataFrame({f'cat={name}': cleaned.apply(lambda xs: name in xs) for name in selected})
            feature_frames.append(cat_df.astype(bool))

    # Categóricas discretas
    for col in ['Price_category', 'Release_season', 'metacritic_cat']:
        if col in df.columns:
            d = pd.get_dummies(df[col].fillna('Unknown').astype(str), prefix=col, prefix_sep='=')
            feature_frames.append(d.astype(bool))

    # Plataforma
    for col, label in [('Windows', 'platform=Windows'), ('Mac', 'platform=Mac'), ('Linux', 'platform=Linux')]:
        if col in df.columns:
            feature_frames.append(pd.DataFrame({label: df[col].fillna(False).astype(bool)}))

    # DLC y descuento (bins)
    if 'DLC count' in df.columns:
        dlc_bin = _bin_series(df['DLC count'], bins=[-0.1, 0, 2, 5, 1_000_000], labels=['0', '1-2', '3-5', '6+']).astype(str)
        feature_frames.append(pd.get_dummies(dlc_bin, prefix='DLC', prefix_sep='=').astype(bool))

    if 'Discount' in df.columns:
        disc = pd.to_numeric(df['Discount'], errors='coerce')
        disc_bin = pd.cut(disc, bins=[-0.1, 0, 25, 50, 75, 1000], labels=['0', '1-25', '26-50', '51-75', '76+'], include_lowest=True).astype(str)
        feature_frames.append(pd.get_dummies(disc_bin, prefix='Discount', prefix_sep='=').astype(bool))

    # Era (reduce cardinalidad de año)
    if 'Release_year' in df.columns:
        year = pd.to_numeric(df['Release_year'], errors='coerce')
        era = pd.cut(year, bins=[0, 2009, 2019, 2029, 9999], labels=['<=2009', '2010-2019', '2020-2029', '>=2030'], include_lowest=True).astype(str)
        feature_frames.append(pd.get_dummies(era, prefix='Release_era', prefix_sep='=').astype(bool))

    features_df = pd.concat(feature_frames, axis=1) if feature_frames else pd.DataFrame(index=df.index)

    # Filtrado de items (para que sea computable y no dominado por items triviales)
    # Importante en datasets grandes: si el número de items es alto y min_support bajo,
    # la cantidad de itemsets frecuentes puede explotar (y FP-Growth/Apriori tardan muchísimo).
    min_item_support = 0.02   # descarta items raros
    max_item_support = 0.50   # descarta items demasiado comunes (poco informativos)
    max_feature_items = 160   # tope de features para no disparar combinatoria

    if not features_df.empty:
        supp = features_df.mean().sort_values(ascending=False)
        keep = supp[(supp >= min_item_support) & (supp <= max_item_support)].head(max_feature_items).index
        features_df = features_df[keep]

    X = pd.concat([features_df, targets_df], axis=1).fillna(False).astype(bool)
    return X, list(features_df.columns), target_items

X, FEATURE_ITEMS, TARGET_ITEMS = build_item_matrix(df)
print('Items features:', len(FEATURE_ITEMS))
print('Items targets :', len(TARGET_ITEMS))
print('X shape      :', X.shape)
TARGET_ITEMS

Items features: 115
Items targets : 5
X shape      : (114174, 120)


['target=review_good',
 'target=sales_good',
 'target=ccu_good',
 'target=reviews_50p_plus',
 'target=reviews_75p_plus']

In [7]:
# Selección de features por métrica de éxito (para acotar el espacio de búsqueda)
# Idea: primero medimos reglas 1→1 (feature → target) y elegimos las features más prometedoras.

TOP_K_FEATURES_PER_TARGET = 40
MIN_FEATURE_SUPPORT = 0.02

# Para targets raros (ej. ventas altas), un min_support fijo (p.ej. 0.02) puede eliminar todas las reglas.
# Por eso ajustamos umbrales según la tasa base del target.
MIN_SUPPORT_CAP = 0.02
MIN_SUPPORT_FLOOR = 0.002

def min_support_for_target(base_rate: float) -> float:
    # Aproximación práctica: permite reglas incluso si el target es raro
    return float(max(MIN_SUPPORT_FLOOR, min(MIN_SUPPORT_CAP, base_rate * 0.25)))

def max_len_for_target(base_rate: float) -> int:
    # Itemsets largos solo cuando el target no es demasiado raro
    return 3 if base_rate >= 0.10 else 2

def min_joint_support_for_target(base_rate: float) -> float:
    # Joint support mínimo para filtrar features candidatas (más bajo si el target es raro)
    return float(max(0.001, min(0.01, base_rate * 0.05)))

def rank_singleton_features(X: pd.DataFrame, feature_items: list[str], target_item: str) -> pd.DataFrame:
    f = X[feature_items].to_numpy(dtype=np.bool_)
    y = X[target_item].to_numpy(dtype=np.bool_)

    feat_support = f.mean(axis=0)
    joint_support = (f & y[:, None]).mean(axis=0)
    base = float(y.mean())
    min_joint = min_joint_support_for_target(base)

    conf = np.divide(joint_support, feat_support, out=np.zeros_like(joint_support), where=feat_support > 0)
    lift = np.divide(conf, base, out=np.zeros_like(conf), where=base > 0)

    out = pd.DataFrame({
        'item': feature_items,
        'support': feat_support,
        'joint_support': joint_support,
        'confidence(item→target)': conf,
        'lift(item→target)': lift,
    })

    out = out[(out['support'] >= MIN_FEATURE_SUPPORT) & (out['joint_support'] >= min_joint)]
    out = out.sort_values(['lift(item→target)', 'confidence(item→target)', 'joint_support'], ascending=False).reset_index(drop=True)
    return out

FEATURE_RANKINGS = {t: rank_singleton_features(X, FEATURE_ITEMS, t) for t in TARGET_ITEMS}
TOP_FEATURES_BY_TARGET = {t: FEATURE_RANKINGS[t].head(TOP_K_FEATURES_PER_TARGET)['item'].tolist() for t in TARGET_ITEMS}

for t in TARGET_ITEMS:
    base = float(X[t].mean())
    print(
        f'\n== {t} == base_rate={base:.3f}  '
        f'min_support≈{min_support_for_target(base):.3f}  '
        f'max_len={max_len_for_target(base)}  '
        f'top_features={len(TOP_FEATURES_BY_TARGET[t])}'
    )
    display(FEATURE_RANKINGS[t].head(10))


== target=review_good == base_rate=0.354  min_support≈0.020  max_len=3  top_features=40


,item,support,joint_support,confidence(item→target),lift(item→target)
0,cat=Remote Play on TV,0.022667,0.015669,0.691267,1.952134
1,tag=Visual Novel,0.051001,0.034360,0.673708,1.902545
2,cat=Steam Trading Cards,0.099418,0.066933,0.673245,1.901238
3,tag=Anime,0.071479,0.047331,0.662174,1.869973
4,cat=Steam Workshop,0.023070,0.015144,0.656416,1.853714
5,tag=Difficult,0.055599,0.035814,0.644140,1.819046
6,tag=Story Rich,0.116874,0.074404,0.636616,1.797798
7,tag=Female Protagonist,0.066381,0.041945,0.631878,1.784417
8,tag=Comedy,0.052332,0.032932,0.629289,1.777106
9,tag=Multiple Endings,0.046815,0.028746,0.614032,1.734021



== target=sales_good == base_rate=0.022  min_support≈0.005  max_len=2  top_features=40


,item,support,joint_support,confidence(item→target),lift(item→target)
0,cat=Remote Play on TV,0.022667,0.004081,0.180062,8.197121
1,tag=Co-op,0.046053,0.007769,0.168695,7.679673
2,cat=Steam Workshop,0.023070,0.003267,0.141610,6.446630
3,tag=Multiplayer,0.088847,0.012297,0.138407,6.300827
4,tag=Open World,0.052481,0.007191,0.137016,6.237507
5,cat=Steam Trading Cards,0.099418,0.011544,0.116113,5.285925
6,cat=In-App Purchases,0.026214,0.002811,0.107250,4.882452
7,cat=Cross-Platform Multiplayer,0.026451,0.002531,0.095695,4.356428
8,cat=Includes level editor,0.022571,0.002137,0.094684,4.310375
9,tag=FPS,0.047086,0.004362,0.092634,4.217060



== target=ccu_good == base_rate=0.042  min_support≈0.011  max_len=2  top_features=40


,item,support,joint_support,confidence(item→target),lift(item→target)
0,cat=Steam Workshop,0.023070,0.005737,0.248671,5.881870
1,tag=Co-op,0.046053,0.009985,0.216812,5.128309
2,tag=Open World,0.052481,0.010353,0.197263,4.665902
3,tag=Multiplayer,0.088847,0.017298,0.194696,4.605192
4,cat=Remote Play on TV,0.022667,0.004379,0.193199,4.569784
5,cat=In-App Purchases,0.026214,0.004782,0.182426,4.314951
6,cat=Steam Trading Cards,0.099418,0.016624,0.167210,3.955050
7,cat=Online Co-op,0.059322,0.009144,0.154141,3.645939
8,cat=Cross-Platform Multiplayer,0.026451,0.004073,0.153974,3.641966
9,cat=Adjustable Difficulty,0.022159,0.003319,0.149802,3.543306



== target=reviews_50p_plus == base_rate=0.510  min_support≈0.020  max_len=3  top_features=40


,item,support,joint_support,confidence(item→target),lift(item→target)
0,cat=Steam Trading Cards,0.099418,0.092508,0.930491,1.822824
1,cat=Remote Play on TV,0.022667,0.018997,0.838099,1.641829
2,tag=Co-op,0.046053,0.038249,0.830544,1.627029
3,tag=Anime,0.071479,0.059252,0.828943,1.623892
4,tag=Difficult,0.055599,0.045466,0.817738,1.601942
5,Discount=76+,0.072512,0.058971,0.813262,1.593175
6,tag=Visual Novel,0.051001,0.041270,0.809205,1.585226
7,tag=Multiplayer,0.088847,0.071785,0.807965,1.582798
8,tag=Psychological Horror,0.053427,0.042996,0.804754,1.576507
9,Release_era=2010-2019,0.262792,0.211248,0.803859,1.574755



== target=reviews_75p_plus == base_rate=0.250  min_support≈0.020  max_len=3  top_features=40


,item,support,joint_support,confidence(item→target),lift(item→target)
0,cat=Steam Trading Cards,0.099418,0.079020,0.794820,3.175997
1,cat=Remote Play on TV,0.022667,0.014863,0.655719,2.620167
2,cat=Steam Workshop,0.023070,0.014040,0.608580,2.431807
3,tag=Co-op,0.046053,0.027038,0.587105,2.345997
4,tag=Multiplayer,0.088847,0.051089,0.575020,2.297704
5,tag=Anime,0.071479,0.038555,0.539395,2.155351
6,tag=Open World,0.052481,0.028027,0.534045,2.133976
7,DLC=1-2,0.123680,0.063193,0.510941,2.041655
8,Discount=76+,0.072512,0.036777,0.507187,2.026653
9,tag=Female Protagonist,0.066381,0.033414,0.503365,2.011379


In [8]:
# FP-Growth (minería por cada target para mantenerlo computable)
# Tip: si todavía tarda mucho, sube MIN_SUPPORT o baja MAX_LEN_FPGROWTH.
freq_fp_by_target = {}

for t in TARGET_ITEMS:
    base = float(X[t].mean())
    min_supp = min_support_for_target(base)
    max_len = max_len_for_target(base)
    feats = TOP_FEATURES_BY_TARGET.get(t, [])
    X_sub = X[feats + [t]] if feats else X[[t]]

    freq = fpgrowth(X_sub, min_support=min_supp, use_colnames=True, max_len=max_len)
    freq = freq.sort_values('support', ascending=False).reset_index(drop=True)
    freq_fp_by_target[t] = freq
    print(f'[{t}] base={base:.3f} min_support={min_supp:.3f} max_len={max_len} items={X_sub.shape[1]} frequent_itemsets={len(freq)}')

# Ejemplo: top itemsets del primer target
first_t = TARGET_ITEMS[0] if TARGET_ITEMS else None
freq_fp_by_target[first_t].head(10) if first_t else None

[target=review_good] base=0.354 min_support=0.020 max_len=3 items=41 frequent_itemsets=456
[target=sales_good] base=0.022 min_support=0.005 max_len=2 items=41 frequent_itemsets=541
[target=ccu_good] base=0.042 min_support=0.011 max_len=2 items=41 frequent_itemsets=303
[target=reviews_50p_plus] base=0.510 min_support=0.020 max_len=3 items=41 frequent_itemsets=552
[target=reviews_75p_plus] base=0.250 min_support=0.020 max_len=3 items=41 frequent_itemsets=306


,support,itemsets
0,0.440976,frozenset({tag=Singleplayer})
1,0.354109,frozenset({target=review_good})
2,0.265735,frozenset({cat=Steam Cloud})
3,0.234642,frozenset({tag=2D})
4,0.226645,"frozenset({target=review_good, tag=Singleplayer})"
5,0.182143,"frozenset({tag=2D, tag=Singleplayer})"
6,0.143360,frozenset({tag=Puzzle})
7,0.142598,frozenset({tag=Atmospheric})
8,0.142379,"frozenset({cat=Steam Cloud, tag=Singleplayer})"
9,0.140216,"frozenset({cat=Steam Cloud, target=review_good})"


In [9]:
# Apriori (también por target; más sensible a explosión combinatoria)
MAX_LEN_APRIORI = 2

freq_ap_by_target = {}

for t in TARGET_ITEMS:
    base = float(X[t].mean())
    min_supp = min_support_for_target(base)
    feats = TOP_FEATURES_BY_TARGET.get(t, [])
    X_sub = X[feats + [t]] if feats else X[[t]]

    freq = apriori(
        X_sub,
        min_support=min_supp,
        use_colnames=True,
        max_len=MAX_LEN_APRIORI,
        low_memory=True,
    )
    freq = freq.sort_values('support', ascending=False).reset_index(drop=True)
    freq_ap_by_target[t] = freq
    print(f'[{t}] base={base:.3f} min_support={min_supp:.3f} items={X_sub.shape[1]} frequent_itemsets={len(freq)}')

first_t = TARGET_ITEMS[0] if TARGET_ITEMS else None
freq_ap_by_target[first_t].head(10) if first_t else None

[target=review_good] base=0.354 min_support=0.020 items=41 frequent_itemsets=259
[target=sales_good] base=0.022 min_support=0.005 items=41 frequent_itemsets=541
[target=ccu_good] base=0.042 min_support=0.011 items=41 frequent_itemsets=303
[target=reviews_50p_plus] base=0.510 min_support=0.020 items=41 frequent_itemsets=283
[target=reviews_75p_plus] base=0.250 min_support=0.020 items=41 frequent_itemsets=217


,support,itemsets
0,0.440976,frozenset({tag=Singleplayer})
1,0.354109,frozenset({target=review_good})
2,0.265735,frozenset({cat=Steam Cloud})
3,0.234642,frozenset({tag=2D})
4,0.226645,"frozenset({target=review_good, tag=Singleplayer})"
5,0.182143,"frozenset({tag=2D, tag=Singleplayer})"
6,0.143360,frozenset({tag=Puzzle})
7,0.142598,frozenset({tag=Atmospheric})
8,0.142379,"frozenset({cat=Steam Cloud, tag=Singleplayer})"
9,0.140216,"frozenset({cat=Steam Cloud, target=review_good})"


In [10]:
# Reglas: forzamos RHS = target (éxito) y LHS = features seleccionadas

def rule_thresholds_for_target(base_rate: float) -> tuple[float, float]:
    # Targets raros: baja confianza mínima pero sube lift mínimo
    if base_rate < 0.06:
        return 0.10, 1.50
    return 0.30, 1.20

def mine_rules_for_target(freq_itemsets: pd.DataFrame, target_item: str, min_confidence=0.35, min_lift=1.10) -> pd.DataFrame:
    if freq_itemsets is None or freq_itemsets.empty:
        return pd.DataFrame()
    rules = association_rules(freq_itemsets, metric='confidence', min_threshold=min_confidence)
    rules = rules[rules['consequents'].apply(lambda s: s == frozenset([target_item]))]
    rules = rules[rules['lift'] >= min_lift]
    rules = rules.sort_values(['lift', 'confidence', 'support'], ascending=False).reset_index(drop=True)
    return rules

def format_rules(rules: pd.DataFrame, top_n=25) -> pd.DataFrame:
    if rules.empty:
        return rules
    out = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(top_n).copy()
    out['antecedents'] = out['antecedents'].apply(lambda s: ', '.join(sorted(list(s))))
    out['consequents'] = out['consequents'].apply(lambda s: ', '.join(sorted(list(s))))
    return out

rules_fp_by_target = {}
rules_ap_by_target = {}
for t in TARGET_ITEMS:
    base = float(X[t].mean())
    min_conf, min_lift = rule_thresholds_for_target(base)
    rules_fp_by_target[t] = mine_rules_for_target(freq_fp_by_target.get(t), t, min_confidence=min_conf, min_lift=min_lift)
    rules_ap_by_target[t] = mine_rules_for_target(freq_ap_by_target.get(t), t, min_confidence=min_conf, min_lift=min_lift)

for t in TARGET_ITEMS:
    base = float(X[t].mean())
    min_conf, min_lift = rule_thresholds_for_target(base)
    print(f'\n== {t} ==')
    print(f'base_rate={base:.3f}  rule_thresholds: min_conf={min_conf:.2f} min_lift={min_lift:.2f}')
    print(f'FP-Growth rules: {len(rules_fp_by_target[t])} | Apriori rules: {len(rules_ap_by_target[t])}')
    display(format_rules(rules_fp_by_target[t], top_n=10))
    display(format_rules(rules_ap_by_target[t], top_n=10))

# ------------------------------
# Top 10 global por algoritmo
# ------------------------------
def _flatten_rules(rules_by_target: dict[str, pd.DataFrame]) -> pd.DataFrame:
    frames = []
    for tgt, r in rules_by_target.items():
        if r is None or r.empty:
            continue
        tmp = r.copy()
        tmp['target'] = tgt
        frames.append(tmp)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

def top10_global(rules_by_target: dict[str, pd.DataFrame], top_n: int = 10) -> pd.DataFrame:
    all_rules = _flatten_rules(rules_by_target)
    if all_rules.empty:
        return all_rules
    all_rules = all_rules.sort_values(['lift', 'confidence', 'support'], ascending=False).reset_index(drop=True)
    out = all_rules[['antecedents', 'consequents', 'target', 'support', 'confidence', 'lift']].head(top_n).copy()
    out['antecedents'] = out['antecedents'].apply(lambda s: ', '.join(sorted(list(s))))
    out['consequents'] = out['consequents'].apply(lambda s: ', '.join(sorted(list(s))))
    return out

print('\n==============================')
print('TOP 10 global (FP-Growth)')
display(top10_global(rules_fp_by_target, top_n=10))

print('\n==============================')
print('TOP 10 global (Apriori)')
display(top10_global(rules_ap_by_target, top_n=10))


== target=review_good ==
base_rate=0.354  rule_thresholds: min_conf=0.30 min_lift=1.20
FP-Growth rules: 126 | Apriori rules: 37


,antecedents,consequents,support,confidence,lift
0,"cat=Steam Trading Cards, tag=2D",target=review_good,0.023683,0.836116,2.361186
1,"DLC=1-2, tag=Story Rich",target=review_good,0.022834,0.807371,2.280008
2,"cat=Steam Cloud, tag=Anime",target=review_good,0.024086,0.801749,2.264134
3,"cat=Steam Trading Cards, tag=Singleplayer",target=review_good,0.043635,0.797248,2.251421
4,"cat=Steam Cloud, tag=Story Rich",target=review_good,0.034868,0.793186,2.239951
5,"DLC=1-2, tag=Atmospheric",target=review_good,0.020880,0.791764,2.235934
6,"cat=Steam Cloud, tag=Female Protagonist",target=review_good,0.021240,0.785298,2.217675
7,"DLC=1-2, tag=2D",target=review_good,0.030226,0.773247,2.183643
8,"cat=Steam Cloud, tag=Funny",target=review_good,0.023578,0.772675,2.182028
9,"cat=Steam Cloud, cat=Steam Trading Cards",target=review_good,0.043539,0.765476,2.161698


,antecedents,consequents,support,confidence,lift
0,tag=Visual Novel,target=review_good,0.034360,0.673708,1.902545
1,cat=Steam Trading Cards,target=review_good,0.066933,0.673245,1.901238
2,tag=Anime,target=review_good,0.047331,0.662174,1.869973
3,tag=Difficult,target=review_good,0.035814,0.644140,1.819046
4,tag=Story Rich,target=review_good,0.074404,0.636616,1.797798
5,tag=Female Protagonist,target=review_good,0.041945,0.631878,1.784417
6,tag=Comedy,target=review_good,0.032932,0.629289,1.777106
7,tag=Multiple Endings,target=review_good,0.028746,0.614032,1.734021
8,tag=Funny,target=review_good,0.053007,0.613855,1.733523
9,tag=Cute,target=review_good,0.066635,0.600047,1.694529



== target=sales_good ==
base_rate=0.022  rule_thresholds: min_conf=0.10 min_lift=1.50
FP-Growth rules: 4 | Apriori rules: 4


,antecedents,consequents,support,confidence,lift
0,tag=Co-op,target=sales_good,0.007769,0.168695,7.679673
1,tag=Multiplayer,target=sales_good,0.012297,0.138407,6.300827
2,tag=Open World,target=sales_good,0.007191,0.137016,6.237507
3,cat=Steam Trading Cards,target=sales_good,0.011544,0.116113,5.285925


...

,antecedents,consequents,support,confidence,lift
0,cat=Steam Trading Cards,target=reviews_75p_plus,0.079020,0.794820,3.175997
1,tag=Co-op,target=reviews_75p_plus,0.027038,0.587105,2.345997
2,tag=Multiplayer,target=reviews_75p_plus,0.051089,0.575020,2.297704
3,tag=Anime,target=reviews_75p_plus,0.038555,0.539395,2.155351
4,tag=Open World,target=reviews_75p_plus,0.028027,0.534045,2.133976
5,DLC=1-2,target=reviews_75p_plus,0.063193,0.510941,2.041655
6,Discount=76+,target=reviews_75p_plus,0.036777,0.507187,2.026653
7,tag=Female Protagonist,target=reviews_75p_plus,0.033414,0.503365,2.011379
8,tag=Story Rich,target=reviews_75p_plus,0.057798,0.494529,1.976075
9,Release_era=2010-2019,target=reviews_75p_plus,0.122497,0.466138,1.862626



TOP 10 global (FP-Growth)


,antecedents,consequents,target,support,confidence,lift
0,tag=Co-op,target=sales_good,target=sales_good,0.007769,0.168695,7.679673
1,tag=Multiplayer,target=sales_good,target=sales_good,0.012297,0.138407,6.300827
2,tag=Open World,target=sales_good,target=sales_good,0.007191,0.137016,6.237507
3,cat=Steam Trading Cards,target=sales_good,target=sales_good,0.011544,0.116113,5.285925
4,tag=Multiplayer,target=ccu_good,target=ccu_good,0.017298,0.194696,4.605192
5,cat=Steam Trading Cards,target=ccu_good,target=ccu_good,0.016624,0.167210,3.955050
6,"cat=Steam Trading Cards, tag=Atmospheric",target=reviews_75p_plus,target=reviews_75p_plus,0.020872,0.946762,3.783138
7,"cat=Steam Trading Cards, tag=Story Rich",target=reviews_75p_plus,target=reviews_75p_plus,0.020626,0.942000,3.764110
8,"cat=Steam Trading Cards, tag=RPG",target=reviews_75p_plus,target=reviews_75p_plus,0.020819,0.896305,3.581517
9,"cat=Steam Trading Cards, tag=Adventure",target=reviews_75p_plus,target=reviews_75p_plus,0.040613,0.872108,3.484832



TOP 10 global (Apriori)


,antecedents,consequents,target,support,confidence,lift
0,tag=Co-op,target=sales_good,target=sales_good,0.007769,0.168695,7.679673
1,tag=Multiplayer,target=sales_good,target=sales_good,0.012297,0.138407,6.300827
2,tag=Open World,target=sales_good,target=sales_good,0.007191,0.137016,6.237507
3,cat=Steam Trading Cards,target=sales_good,target=sales_good,0.011544,0.116113,5.285925
4,tag=Multiplayer,target=ccu_good,target=ccu_good,0.017298,0.194696,4.605192
5,cat=Steam Trading Cards,target=ccu_good,target=ccu_good,0.016624,0.167210,3.955050
6,cat=Steam Trading Cards,target=reviews_75p_plus,target=reviews_75p_plus,0.079020,0.794820,3.175997
7,cat=Co-op,target=ccu_good,target=ccu_good,0.011412,0.118401,2.800556
8,cat=Multi-player,target=ccu_good,target=ccu_good,0.017902,0.100839,2.385157
9,DLC=1-2,target=ccu_good,target=ccu_good,0.012402,0.100276,2.371853
